In [ ]:
"""
<타이타닉 데이터를 이용하여 생존율 예측 실습>
✨ 사용 데이터 : 03_exe/ 폴더 내에 csv 파일 사용
 - 머신러닝 분류분석, 딥러닝 DNN 사용하여 가장 좋은 모델 찾기
 - 조별 실습
   -> 제출은 각자 이름으로 파일명 만드셔서 구글드라이브 15day에 제출
   -> 제출 폴더 : 04_타이타닉_생존_예측(개별제출)
   -> 제출 파일명 : 02_(홍길동)Titanic_exe.ipynb
"""

In [98]:
### 사용할 라이브러리 불러들이기

# 데이터 조작을 위한 라이브러리
import pandas as pd
import numpy as np

# 기본 시각화 라이브러리
import matplotlib.pyplot as plt

# 통계적 시각화 라이브러리(보통 기본 시각화보다 향상된 라이브러리라고 생각하면 됨)
import seaborn as sns

# 시각화 시 그래프의 너비/높이를 기본 설정하기(단위 = 인치)
plt.rcParams["figure.figsize"] = (12, 5)

# 시각화시 한글 깨짐 처리를 위한 폰트 지정
plt.rcParams["font.family"] = "Malgun Gothic"

# 시각화시 마이너스(-) 기호 깨짐 방지 처리
plt.rcParams["axes.unicode_minus"] = False

# 상관관계 검정(scipy : 통계 라이브러리)
from scipy.stats import spearmanr, pearsonr

# 훈련/검증/테스트 데이터로 분리하는 함수
from sklearn.model_selection import train_test_split

# 데이터스케일링 표준화 클래스
from sklearn.preprocessing import StandardScaler

# 사용할 훈련모델 클래스
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier

# 분류분석 오차확인(평가) 라이브러리
# - 오차행렬표, 오차행렬도(그래프)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 모델 예측 평가지표 확인
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 분류모델 평가지표(선정기준)
# - 정확도, 정밀도, 재현율, f1-score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 군집 모델
from sklearn.cluster import KMeans, DBSCAN

# 튜닝 클래스
from sklearn.model_selection import GridSearchCV

# 지도-분류분석 평가
from sklearn.metrics import accuracy_score, confusion_matrix

# 군집분석 평가
from sklearn.metrics import silhouette_score, adjusted_rand_score

In [99]:
# 파일 불러오기
file_path =("./03_exe/(실습)타이타닉_분류_데이터셋.csv")

# 파일 정보 확인
df_a = pd.read_csv(file_path)
df_a.info()
display(df_a.describe())
df_a.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    object 
 8   class        891 non-null    object 
 9   who          891 non-null    object 
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    object 
 12  embark_town  889 non-null    object 
 13  alive        891 non-null    object 
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(7)
memory usage: 92.4+ KB


,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [100]:
# 데이터 파일에서 사용할 컬럼 추출
df_b = df_a[["survived", "pclass", "age", "sex", "who", "sibsp", "parch", "embarked"]]

# 데이터 추출한 파일 정보 확인
df_b.info()
display(df_b.describe())
df_b.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   survived  891 non-null    int64  
 1   pclass    891 non-null    int64  
 2   age       714 non-null    float64
 3   sex       891 non-null    object 
 4   who       891 non-null    object 
 5   sibsp     891 non-null    int64  
 6   parch     891 non-null    int64  
 7   embarked  889 non-null    object 
dtypes: float64(1), int64(4), object(3)
memory usage: 55.8+ KB


,survived,pclass,age,sibsp,parch
count,891.000000,891.000000,714.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594
std,0.486592,0.836071,14.526497,1.102743,0.806057
min,0.000000,1.000000,0.420000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000
50%,0.000000,3.000000,28.000000,0.000000,0.000000
75%,1.000000,3.000000,38.000000,1.000000,0.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000


,survived,pclass,age,sex,who,sibsp,parch,embarked
0,0,3,22.0,male,man,1,0,S
1,1,1,38.0,female,woman,1,0,C
2,1,3,26.0,female,woman,0,0,S
3,1,1,35.0,female,woman,1,0,S
4,0,3,35.0,male,man,0,0,S


### 이상치 결측치 확인

In [101]:
# 컬럼 추출한 파일의 컬럼 내 이상치 및 결측치 확인
df_b.info()
df_b.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   survived  891 non-null    int64  
 1   pclass    891 non-null    int64  
 2   age       714 non-null    float64
 3   sex       891 non-null    object 
 4   who       891 non-null    object 
 5   sibsp     891 non-null    int64  
 6   parch     891 non-null    int64  
 7   embarked  889 non-null    object 
dtypes: float64(1), int64(4), object(3)
memory usage: 55.8+ KB


survived      0
pclass        0
age         177
sex           0
who           0
sibsp         0
parch         0
embarked      2
dtype: int64

In [ ]:
# who컬럼에서 child와 성인남녀를 분리  (이상치를 먼저 중앙값(child를 빼고 남녀 성인 나이의 평균)으로 변경하고 결측치를 중앙값으로 채운다.) 
age_median =df_b.groupby("who")["age"].transform("median")


In [103]:
## 독립된 데이터로 복사
df_b =df_b.copy()

In [104]:
# 결측치를 중앙값으로 채우기
df_b["age"]=df_b["age"].fillna(age_median)

In [105]:
# 결과확인
df_b["age"].isnull().sum()

np.int64(0)

In [106]:
df_b.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   survived  891 non-null    int64  
 1   pclass    891 non-null    int64  
 2   age       891 non-null    float64
 3   sex       891 non-null    object 
 4   who       891 non-null    object 
 5   sibsp     891 non-null    int64  
 6   parch     891 non-null    int64  
 7   embarked  889 non-null    object 
dtypes: float64(1), int64(4), object(3)
memory usage: 55.8+ KB


In [ ]:

"""
#age 정수타입으로 정리하기 

df_b["age"] = df_b["age"].astype("int")
df_b.info()
df_b.head(5)

"""

In [107]:

# 엠바크 결측치 채우기 
df_b["embarked"] = df_b["embarked"].fillna(df_b["embarked"].mode()[0])

In [108]:
df_b.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   survived  891 non-null    int64  
 1   pclass    891 non-null    int64  
 2   age       891 non-null    float64
 3   sex       891 non-null    object 
 4   who       891 non-null    object 
 5   sibsp     891 non-null    int64  
 6   parch     891 non-null    int64  
 7   embarked  891 non-null    object 
dtypes: float64(1), int64(4), object(3)
memory usage: 55.8+ KB


In [ ]:
# 데이터 who의 데이터를 personal에 변수로 
# age, sex, who를 이용하여 info 컬럼 생성
# 컬럼 이름 소문자로 통일하고 
df_b.columns = df_b.columns.str.lower()

# info 생성 함수
def create_info(row):
    age = row["age"]
    sex = row["sex"]

    if age <= 17:
        return "child"
    elif age >= 61:
        return "old"
    elif sex == "male":
        return "man"
    else:
        return "woman"

print(df_b)

     survived  pclass  sibsp  parch  personal_child  personal_elderly  \
0           0       3      1      0               0                 0   
1           1       1      1      0               0                 0   
2           1       3      0      0               0                 0   
3           1       1      1      0               0                 0   
4           0       3      0      0               0                 0   
..        ...     ...    ...    ...             ...               ...   
886         0       2      0      0               0                 0   
887         1       1      0      0               0                 0   
888         0       3      1      2               0                 0   
889         1       1      0      0               0                 0   
890         0       3      0      0               0                 0   

     personal_man  personal_woman  embarked_c  embarked_q  embarked_s personal  
0               1               0         

In [ ]:
#④ 데이터 분리 60% / 검증 20% / 테스트 20%


